# XIOS — adaptive-depth chat model

Runs end to end on a **free Colab T4**. The experiments are ordered so the cheapest way to be proven wrong comes first.

1. **Invariants** — the architecture is self-consistent (seconds).
2. **The floor** — XIOS matches a dense baseline on tasks the baseline solves. If it fails here, nothing else matters.
3. **Calibration** — find a task width the model can do *one* step of, before measuring how many it can chain.
4. **The real test** — depth extrapolation against a parameter-matched transformer.
5. **Natural language** — the same A/B on text.
6. **Does it fit on a bad PC** — int4 + disk memory, measured.

Runtime → Change runtime type → **T4 GPU** first.

In [ ]:
#@title STEP 1 - Setup (clone + install). Run this first.
import os, sys, pathlib

REPO = pathlib.Path('/content/xios')
if not REPO.exists():
    !git clone https://github.com/GooberNiko/xios.git /content/xios
sys.path.insert(0, str(REPO))
os.chdir(REPO)

!pip -q install -r requirements.txt

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
#@title STEP 2 - Preflight. ~30s; catches device/dtype bugs before you waste a run.
!python preflight.py

In [ ]:
#@title STEP 3 - Invariants (~1 min, all 7 test modules)
!python run_tests.py --fast

## 2. The floor

XIOS must match a dense baseline on a task the baseline solves outright. This caught two real bugs in the looped core: renormalising the residual stream every iteration, and textbook ACT output-mixing blending under-computed states into the answer. Both cost double-digit accuracy.

In [ ]:
!python tests/test_learns.py

## 3. Calibrate the task width

Each task has a *width* knob (table size) separate from its *depth* knob (chained steps). If the model cannot do **one** step at the chosen width, the depth curve is noise — that mistake made an earlier version of this experiment read as a flat line at chance. Find the largest width that works, then use it below.

In [ ]:
import torch
from xios.config import get_config
from xios.model import XiosChat
from xios.tasks import TASK_VOCAB, calibrate_width

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = get_config('nano', vocab_size=TASK_VOCAB, max_seq_len=128,
                 max_iters=12, target_depth=4.0, attn_window=96)

best = calibrate_width(lambda: XiosChat(cfg), 'hop', seq_len=96,
                       widths=(6, 8, 10, 12), steps=800, device=dev)
print(f'\nuse --width {best}')

## 4. The real test — depth extrapolation

Both models train on 1–4 hop problems and are tested to 9. A fixed-depth transformer cannot compose more dependent operations than its depth permits, so it should fall off a cliff past its limit. If adaptive depth works, XIOS degrades more gracefully **and** its mean depth rises with difficulty — which nothing in the loss asks it to do.

Set `--width` from the cell above. ~20–30 min on a T4.

In [ ]:
# `perm` composes a FRESH permutation each step, so difficulty is
# genuinely monotone in step count. `hop` follows ONE table k times and
# lands at k mod cycle_length -- measured non-monotone, and a flat depth
# curve on it means nothing. Use perm for the headline experiment.
!python train/ab_experiment.py \n    --task perm --width 4 --preset nano \n    --steps 8000 --batch-size 128 --seq-len 64 --lr 1e-3 \n    --train-max-steps 4 --eval-max-steps 9 \n    --max-iters 12 --target-depth 6 --fixed-depth --curriculum-frac 0.35 \n    --eval-n 1024 --log-every 500 --out runs/ab_perm

In [ ]:
#@title Plot: accuracy and allocated depth vs difficulty
import json, matplotlib.pyplot as plt

res = json.loads(open('runs/ab_perm/results.json').read())
t = res['table']
steps = [r['steps'] for r in t]
trained_to = res['config']['train_max_steps']

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(steps, [r.get('baseline', 0) for r in t], 'o-', label='baseline transformer')
ax[0].plot(steps, [r.get('xios', 0) for r in t], 's-', label='XIOS')
ax[0].axvline(trained_to + .5, ls='--', c='k', lw=1)
ax[0].text(trained_to + .6, .85, 'extrapolation →', fontsize=9)
ax[0].set_xlabel('required sequential steps'); ax[0].set_ylabel('exact match')
ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title('accuracy')

ax[1].plot(steps, [r.get('xios_depth', 0) for r in t], 'd-', c='C2')
ax[1].axvline(trained_to + .5, ls='--', c='k', lw=1)
ax[1].set_xlabel('required sequential steps'); ax[1].set_ylabel('mean iterations used')
ax[1].grid(alpha=.3)
ax[1].set_title('depth the controller chose (nothing asked it to)')
plt.tight_layout(); plt.show()

In [ ]:
#@title Repeat on the other depth-sensitive tasks
# hop caps at width-1 steps (single cycle); the harness clamps and warns.
for task, width in [('mod', 5), ('parity', 0), ('hop', 8)]:
    print('=' * 70, '
', task, '
', '=' * 70)
    w = f'--width {width}' if width else ''
    !python train/ab_experiment.py --task {task} {w} --preset nano \n        --steps 6000 --batch-size 128 --seq-len 96 --lr 1e-3 \n        --train-max-steps 4 --eval-max-steps 9 \n        --max-iters 12 --target-depth 6 --fixed-depth --curriculum-frac 0.35 \n        --eval-n 1024 --log-every 500 --out runs/ab_{task}

## 5. Natural language

Perplexity averages over easy and hard tokens, so the effect is smaller here by construction. The question is whether XIOS holds parity while keeping its depth headroom available — and whether the depth histogram looks sensible (function words cheap, content words expensive).

In [ ]:
!pip -q install datasets transformers safetensors
!python train/train_text.py --dataset roneneldan/TinyStories --preset nano \
    --steps 6000 --batch-size 48 --seq-len 512 --vocab 8192 \
    --max-iters 12 --target-depth 4 --out runs/text

## 6. Does it fit on a bad PC?

In [ ]:
!python -m xios.cli info
!python -m xios.cli quantize --preset small
!python -m xios.cli bench --preset micro --prompt-len 256 --n-tokens 64

In [ ]:
#@title Export the disk-resident memory, seriate its layout, measure page reads
import torch
from xios.config import get_config
from xios.model import XiosChat
from xios.memory.dam import DiskAssociativeMemory

cfg = get_config('small')
m = XiosChat(cfg)          # RAM-backed values, as during training
for mod in m.modules():
    if isinstance(mod, DiskAssociativeMemory):
        x = torch.randn(1, 256, cfg.dim)
        q = mod.q_norm(mod.q_proj(x).reshape(-1, mod.n_heads, 2, mod.key_dim))
        i1, i2 = mod.logical_codes(q)
        print('page reads per token:', mod.locality_report(i1, i2))
        mod.optimize_layout('sorted_morton')
        print('exported:', mod.export('/content/xios_memory'))
        break

## 7. Test-time depth scaling — the headline result

The same weights, run longer. On CPU this took 12-composition problems from
4.7% (chance) at the trained 6 iterations to **97.7%** at 16. Extra iterations
cost ~no DRAM bandwidth on a cache-resident core, so this is the axis where a
small local model can outbuy a dense one. Does it hold at scale?

In [ ]:
!python train/test_time_depth.py \n    --ckpt runs/ab_perm/xios.pt --task perm --width 4 \n    --trained-depth 6 --depths 2,4,6,8,12,16,24,32 \n    --problem-steps 4,8,10,12,14,16,20 \n    --seq-len 192 --eval-n 512 --out runs/ttd

In [ ]:
#@title Plot: accuracy vs test-time iterations
import json, matplotlib.pyplot as plt
g = json.loads(open('runs/ttd/grid.json').read())
for s in g['steps']:
    ys = [g['grid'][f"{s}_{d}"] for d in g['depths']]
    plt.plot(g['depths'], ys, 'o-', label=f'{s} steps')
plt.axvline(g['args']['trained_depth'], ls='--', c='k', lw=1)
plt.text(g['args']['trained_depth']+.2, .05, 'trained depth', fontsize=8)
plt.xlabel('iterations at inference'); plt.ylabel('exact match')
plt.title('thinking longer with the same weights')
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()